# 01 · Definición del problema y análisis ético

**Proyecto Integrador:** Análisis predictivo de riesgo de crédito en instituciones financieras.

| ID del enunciado | Qué cubre este notebook | Sección |
|---|---|---|
| **F1** | Fase 1 de la metodología: definir el problema | 1, 2 |
| **E5** | Enfoque en la solución de un problema dentro de un contexto | 1 |
| **IA1.1** | Explorar datasets financieros y definir el problema de riesgo de crédito | 2, 3, 6 |
| **IA1.3** | Análisis ético: sesgos, variables sensibles e impacto en las personas | 5 |

Estructura: **1. Contexto · 2. Formulación · 3. Datasets · 4. Métricas · 5. Ética · 6. Carga inicial · Conclusiones**

In [1]:
%matplotlib inline
import warnings
warnings.filterwarnings("ignore")
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

def buscar_raiz(inicio: Path = Path.cwd()) -> Path:
    """Sube por los directorios hasta encontrar db/credit_risk.csv (rutas relativas, sin rutas absolutas)."""
    for carpeta in [inicio, *inicio.parents]:
        if (carpeta / "db" / "credit_risk.csv").exists():
            return carpeta
    raise FileNotFoundError("No se encontró db/credit_risk.csv; ejecute el notebook desde Back/notebooks/")

PROJECT_ROOT = buscar_raiz()
RAW_DATA_PATH = PROJECT_ROOT / "db" / "credit_risk.csv"      # equivalente a config.DATA_PATH
PROCESSED_DIR = PROJECT_ROOT / "db" / "processed"
TARGET, ID_COL, TEXT_COL = "default", "id_cliente", "texto_solicitud"

sns.set_theme(style="whitegrid", palette="deep")
plt.rcParams.update({"figure.dpi": 100, "axes.titleweight": "bold"})
pd.options.display.max_columns = 50
pd.options.display.width = 160
pd.options.display.float_format = "{:,.3f}".format
print("Raíz del proyecto (relativa al notebook):", Path("..", "..").as_posix())

Raíz del proyecto (relativa al notebook): ../..


## 1. Contexto de negocio

Una institución financiera colombiana recibe miles de solicitudes de crédito de consumo al mes. Antes de aprobar un crédito necesita estimar la **probabilidad de impago (PD, *probability of default*)** del solicitante.

- **Usuario del modelo:** el **analista de crédito**, que recibe la PD y una banda de riesgo como apoyo (el modelo no decide solo).
- **Decisión que apoya:** **aprobar**, **rechazar** o **enviar a revisión manual** la solicitud.
- **Costo de cada error:**

| Error | Qué ocurre | Costo para el negocio | Costo para la persona |
|---|---|---|---|
| **Falso negativo** (FN): se predice "paga" y el cliente **incumple** | Se aprueba un crédito que no se recupera | **Pérdida por impago** (capital + intereses + cobranza). Es el error **más caro**. | Sobreendeudamiento, reporte negativo en centrales |
| **Falso positivo** (FP): se predice "impago" y el cliente **habría pagado** | Se rechaza a un buen cliente | Costo de oportunidad (margen financiero perdido) | **Exclusión financiera** injusta |

Por eso el umbral de decisión no será 0.5 por defecto: se elegirá en el paso 8 con una matriz de costos donde **FN > FP**.

## 2. Formulación técnica

| Elemento | Definición |
|---|---|
| Tipo de problema | **Clasificación binaria supervisada** |
| Unidad de análisis | Una solicitud de crédito (un cliente) |
| Variable objetivo | `default` (1 = impago, 0 = pagó) |
| Variables de entrada | Demográficas, laborales, financieras, de buró y el texto libre de la solicitud |
| Salida del modelo | Una **probabilidad** de impago en [0, 1] **más una banda de riesgo**: Bajo, Medio o Alto |
| Bandas iniciales (se recalibran en el paso 8) | Bajo: p < 0.20 · Medio: 0.20 ≤ p < 0.50 · Alto: p ≥ 0.50 |
| Excluidas como predictor | `id_cliente` (identificador) y `sexo` (sensible; solo se usa para auditar) |

In [2]:
# Representación de la salida esperada: probabilidad -> banda de riesgo
BANDAS_RIESGO = {"Bajo": (0.0, 0.20), "Medio": (0.20, 0.50), "Alto": (0.50, 1.0)}

def banda_riesgo(p: float) -> str:
    """Convierte una probabilidad de impago en una banda de riesgo."""
    for banda, (lo, hi) in BANDAS_RIESGO.items():
        if lo <= p < hi or (banda == "Alto" and p == 1.0):
            return banda
    raise ValueError("La probabilidad debe estar en [0, 1]")

for p in [0.05, 0.35, 0.80]:
    print(f"PD = {p:.2f} -> banda {banda_riesgo(p)}")

PD = 0.05 -> banda Bajo
PD = 0.35 -> banda Medio
PD = 0.80 -> banda Alto


## 3. Exploración de datasets financieros

**Fuente usada:** `db/credit_risk.csv` (ver `docs/diccionario_datos.md`). Es un dataset **sintético realista** de ~12 mil solicitudes, generado con semilla 42, ambientado en Colombia (montos en COP, puntaje de buró tipo DataCrédito) e inspirado en la estructura de *German Credit* y *Default of Credit Card Clients* (UCI). Incluye imperfecciones intencionales (nulos, duplicados, atípicos, edades inválidas) y una columna de **texto libre** para NLP.

| Dataset | Fuente | Filas × columnas | Objetivo | Ventajas | Limitaciones para este proyecto |
|---|---|---|---|---|---|
| **credit_risk.csv** (elegido) | Sintético propio (`db/generacion`) | ~12 000 × 21 | `default` | Contexto colombiano (COP, buró), variables demográficas + financieras + **texto**, imperfecciones para practicar limpieza, sin datos personales reales | Sintético: las relaciones son plausibles pero no observadas en un banco real |
| German Credit | UCI ML Repository | 1 000 × 20 | Buen/mal pagador | Clásico, muy citado, incluye variables sensibles | Muy pequeño, codificado (A11, A12…), moneda DM, sin texto |
| Default of Credit Card Clients | UCI (Taiwán, 2005) | 30 000 × 24 | Impago del mes siguiente | Real y de buen tamaño | Solo tarjetas de crédito; sin texto; contexto taiwanés |
| Give Me Some Credit | Kaggle (2011) | 150 000 × 11 | Mora grave a 2 años | Grande, real | Pocas variables, sin demográficas ricas ni texto |

**Justificación de la elección:** el dataset propio es el único que permite cubrir **todas** las unidades del curso con un solo conjunto de datos: limpieza (tiene nulos, duplicados y atípicos), codificación (≥ 3 nominales y 1 ordinal), NLP (columna de texto en español), auditoría de equidad (variables sensibles) y un contexto de negocio colombiano coherente. Además, al ser sintético, **no expone datos personales reales** (Ley 1581 de 2012). Los datasets públicos se mencionan como referencia y para comparar el orden de magnitud del desempeño esperado (AUC 0.70–0.85 en la literatura).

## 4. Métricas de éxito

| Métrica | Rol | Por qué |
|---|---|---|
| **ROC-AUC** | **Principal** | Mide la capacidad de ordenar clientes por riesgo, independiente del umbral. Estándar en *scoring* crediticio. |
| Recall de la clase impago | Secundaria | Proporción de impagos detectados: controla el error más caro (FN). |
| F1 (clase impago) | Secundaria | Balance entre precisión y recall con clases desbalanceadas. |
| **KS** (Kolmogorov-Smirnov) | Secundaria | Máxima separación entre las distribuciones de score de buenos y malos; métrica regulatoria habitual en banca. |
| **Brier score** | Secundaria | Calidad de la **probabilidad** (calibración): el analista usa la PD, no solo la clase. |

**Meta mínima:** **ROC-AUC ≥ 0.70 en el conjunto de prueba**, superando claramente a la línea base (`DummyClassifier`, AUC = 0.5).

## 5. Análisis ético

### 5.1 Variables sensibles
La siguiente tabla queda **definida aquí y se reutiliza en el paso 8** (auditoría de equidad). Se guarda en la variable `VARIABLES_SENSIBLES`.

In [3]:
VARIABLES_SENSIBLES = pd.DataFrame([
    {"variable": "sexo", "tipo": "protegida (género)", "uso_en_modelo": "NO (solo auditoría)",
     "grupos_auditoria": "F vs M", "riesgo": "Discriminación directa por género"},
    {"variable": "edad", "tipo": "protegida (edad)", "uso_en_modelo": "Sí, auditada",
     "grupos_auditoria": "18-25, 26-35, 36-45, 46-60, >60", "riesgo": "Penalizar a jóvenes o adultos mayores"},
    {"variable": "estado_civil", "tipo": "sensible (situación familiar)", "uso_en_modelo": "Sí, auditada",
     "grupos_auditoria": "Soltero, Casado, Union_libre, Divorciado, Viudo", "riesgo": "Discriminación por situación familiar"},
    {"variable": "personas_a_cargo", "tipo": "proxy potencial", "uso_en_modelo": "Sí, vigilada",
     "grupos_auditoria": "0, 1-2, 3+", "riesgo": "Proxy de género/estado civil"},
    {"variable": "tipo_vivienda", "tipo": "proxy socioeconómico", "uso_en_modelo": "Sí, vigilada",
     "grupos_auditoria": "Propia, Arriendo, Familiar", "riesgo": "Proxy de nivel socioeconómico"},
    {"variable": "texto_solicitud", "tipo": "texto libre", "uso_en_modelo": "Sí (NLP), vigilada",
     "grupos_auditoria": "-", "riesgo": "Puede revelar datos sensibles o sesgos de redacción"},
]).set_index("variable")

# Columnas que nunca entran al modelo como predictor
EXCLUIDAS_DEL_MODELO = ["id_cliente", "sexo"]
VARIABLES_SENSIBLES

,tipo,uso_en_modelo,grupos_auditoria,riesgo
variable,,,,
sexo,protegida (género),NO (solo auditoría),F vs M,Discriminación directa por género
edad,protegida (edad),"Sí, auditada","18-25, 26-35, 36-45, 46-60, >60",Penalizar a jóvenes o adultos mayores
estado_civil,sensible (situación familiar),"Sí, auditada","Soltero, Casado, Union_libre, Divorciado, Viudo",Discriminación por situación familiar
personas_a_cargo,proxy potencial,"Sí, vigilada","0, 1-2, 3+",Proxy de género/estado civil
tipo_vivienda,proxy socioeconómico,"Sí, vigilada","Propia, Arriendo, Familiar",Proxy de nivel socioeconómico
texto_solicitud,texto libre,"Sí (NLP), vigilada",-,Puede revelar datos sensibles o sesgos de reda...


### 5.2 Riesgos identificados

1. **Sesgo histórico en las etiquetas:** si en el pasado se negó crédito a ciertos grupos, el `default` observado refleja esas decisiones y el modelo puede perpetuarlas.
2. **Discriminación indirecta por *proxies*:** aunque se retire `sexo`, otras variables (personas a cargo, estado civil, tipo de vivienda, redacción del texto) pueden reconstruirlo.
3. **Falta de explicabilidad:** un rechazo sin explicación vulnera el derecho del solicitante a conocer los motivos y dificulta la supervisión (Superintendencia Financiera).
4. **Privacidad:** los datos de ingresos, deudas y buró son datos personales; su uso exige finalidad legítima, consentimiento y seguridad.

### 5.3 Medidas de mitigación

| Medida | Cómo se implementa | Paso |
|---|---|---|
| No usar `sexo` como predictor | Se excluye en `feature_list.json`; solo se usa para auditar | 3, 8 |
| Medir la **equidad** por grupo | Tasa de aprobación, TPR y FPR por grupo; ***disparate impact* ≥ 0.8** (regla de los cuatro quintos) | 8 |
| Explicar el modelo | Importancia global de variables y explicación de casos individuales; factores principales en la API | 5, 8, 10 |
| Anonimizar | `id_cliente` es un código sin datos personales (`CLI_xxxxxx`); no hay nombres ni documentos | 1 |
| Humano en el ciclo | La banda "Medio" se envía a revisión manual del analista | 8 |

### 5.4 Referencia normativa
- **Ley 1581 de 2012 (Colombia)** y Decreto 1377 de 2013: protección de datos personales (principios de finalidad, libertad/consentimiento, seguridad, confidencialidad y acceso).
- **Ley 1266 de 2008 (Habeas Data financiero):** regula el uso de la información de centrales de riesgo.
- **Principios de IA responsable** (OCDE 2019; marco ético de IA de Colombia, 2021): equidad, transparencia, explicabilidad, responsabilidad y supervisión humana.

## 6. Carga inicial del dataset
Solo para confirmar su forma y la tasa de `default`. La exploración completa está en el notebook 02.

In [4]:
df = pd.read_csv(RAW_DATA_PATH)
print(f"Filas: {df.shape[0]:,} | Columnas: {df.shape[1]}")
print("Columnas:", list(df.columns))
tasa_default = df[TARGET].mean()
print(f"Tasa de default: {tasa_default:.2%}")
df[TARGET].value_counts().rename(index={0: "Pagó (0)", 1: "Impago (1)"}).to_frame("clientes")

Filas: 12,144 | Columnas: 21
Columnas: ['id_cliente', 'edad', 'sexo', 'estado_civil', 'nivel_educativo', 'situacion_laboral', 'antiguedad_laboral_anios', 'personas_a_cargo', 'tipo_vivienda', 'ingreso_mensual', 'monto_credito', 'plazo_meses', 'tasa_interes_anual', 'proposito', 'num_creditos_activos', 'moras_ultimos_12m', 'relacion_deuda_ingreso', 'puntaje_buro', 'uso_cupo_tarjetas', 'texto_solicitud', 'default']
Tasa de default: 35.63%


,clientes
default,
Pagó (0),7817
Impago (1),4327


In [5]:
# Vista rápida de equidad "en bruto": tasa de default observada por variable sensible
# (no implica sesgo del modelo; sirve como línea base para la auditoría del paso 8)
rango_edad = pd.cut(df["edad"].where(df["edad"].between(18, 100)), [17, 25, 35, 45, 60, 100],
                    labels=["18-25", "26-35", "36-45", "46-60", ">60"])
for nombre, serie in {"sexo": df["sexo"], "rango_edad": rango_edad, "estado_civil": df["estado_civil"]}.items():
    tabla = df.groupby(serie, observed=True)[TARGET].agg(clientes="size", tasa_default="mean")
    print(f"\n--- {nombre} ---")
    print(tabla.round(3))


--- sexo ---
      clientes  tasa_default
sexo                        
F         5871         0.359
M         6273         0.354

--- rango_edad ---
       clientes  tasa_default
edad                         
18-25      1505         0.407
26-35      1972         0.385
36-45      1939         0.372
46-60      2939         0.342
>60        3764         0.325

--- estado_civil ---


              clientes  tasa_default
estado_civil                        
Casado            4093         0.355
Divorciado        1068         0.378
Soltero           4569         0.353
Union_libre       1813         0.347
Viudo              601         0.379


## Conclusiones

- El problema queda definido como **clasificación binaria supervisada** que estima la **probabilidad de impago** (`default`) y la traduce a una **banda de riesgo** (Bajo / Medio / Alto) para apoyar al analista en aprobar, rechazar o revisar.
- El error más costoso es el **falso negativo** (aprobar a quien no paga); el umbral se fijará con una matriz de costos en el paso 8.
- Se eligió el dataset sintético `credit_risk.csv` (~12 mil filas, 21 columnas) frente a German Credit, Default of Credit Card Clients y Give Me Some Credit porque cubre todas las unidades del curso (limpieza, codificación, NLP, equidad) en contexto colombiano y sin datos personales reales.
- La **tasa de default es ≈ 35.6 %**: clases moderadamente desbalanceadas, por lo que se usarán división estratificada, `class_weight` y métricas como ROC-AUC, recall, F1, KS y Brier (meta: **AUC ≥ 0.70** en test).
- Ética: `sexo` **no** será predictor; edad y estado civil se auditan; la tabla `VARIABLES_SENSIBLES` se reutiliza en el paso 8 para medir tasa de aprobación, TPR, FPR y *disparate impact* (≥ 0.8). En los datos crudos la tasa de default es casi igual entre F y M, y mayor en los jóvenes de 18-25 años, lo que obliga a vigilar el efecto de la edad.